# 03 - Champion Selection & Experiment Balance Check

This notebook:
1. Dynamically queries `db.sqlite3` for all completed LLaMEA experiments present in the database.
2. Displays an **experiment summary** grouped by problem ID, dimension, noise level, prompt strategy, and LLM model family.
3. Selects separate **Clean** ($\sigma=0.0$) and **Noisy** ($\sigma>0.0$) champion algorithms per condition (lowest ground-truth error across iterations, with evaluation efficiency as tie-breaker).
4. Exports `data/champions.json` ready for multi-run empirical evaluation in **Notebook 04** (`04_evaluate_champions.ipynb`).


In [ ]:
import sys
from pathlib import Path
from IPython.display import display

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from shared.config import DATA_DIR
from benchmarking import ChampionSelectionService

service = ChampionSelectionService()
CHAMPIONS_PATH = DATA_DIR / 'champions.json'
print(f'Champions Output Path: {CHAMPIONS_PATH}')


## 1. Experiment Balance Check

In [ ]:
# Query all completed experiments dynamically via BenchmarkDataService
summary, total_completed = service.get_experiment_balance()

if summary.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

print(f'Total completed experiments in database: {total_completed}')
print('\n=== Completed Experiments Summary (Problem x Dim x Noise x Strategy) ===')
display(summary) if 'display' in globals() else print(summary.to_string(index=False))


## 2. Select & Export Problem-Specific Champions (Dynamically)

In [ ]:
# Query, format, and export champions dynamically via BenchmarkDataService
champions_dict, df_champions = service.export_champions(CHAMPIONS_PATH)

if not champions_dict:
    raise RuntimeError('No completed iterations found in database.')

print('=== Dimension-Specific Champions (Dynamically Discovered from DB) ===\n')
for model_name, grp in df_champions.groupby('model'):
    print(f'🔹 Model: {model_name} ({len(grp)} champions)')

total_champs = len(df_champions)
n_models = df_champions['model'].nunique()
print(f'\n✨ Exported {total_champs} dimension-specific champion(s) across {n_models} DB model(s) to {CHAMPIONS_PATH}')
display(df_champions[['model', 'key', 'problem_id', 'dim', 'mode', 'prompt_strategy', 'algorithm_name', 'final_error']].head(15))
